# **Data Cleaning Notebook**

In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [23]:
import os 
import sys
sys.path.append(os.path.abspath('Titanic-Survival-Dataset/src'))
os.listdir('Titanic-Survival-Dataset/src')

['utils.py', '__pycache__']

In [24]:
df=pd.read_csv('Titanic-Survival-Dataset/Data/raw/train.csv')
df.head(10)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C


In [25]:
from utils import data_audit
audit=data_audit(df)
audit.head(12)

,Data Type,Missing Values,Unique Values,Percentage Missing,zero variance columns,Duplicate rows
PassengerId,int64,0,891,0.000000,False,0
Survived,int64,0,2,0.000000,False,0
Pclass,int64,0,3,0.000000,False,0
Name,str,0,891,0.000000,False,0
Sex,str,0,2,0.000000,False,0
Age,float64,177,88,19.865320,False,0
SibSp,int64,0,7,0.000000,False,0
Parch,int64,0,7,0.000000,False,0
Ticket,str,0,681,0.000000,False,0
Fare,float64,0,248,0.000000,False,0


1. *Age* attribute has an overall missing value percentage as 19.86%
2. *Cabin* column has the highest missing values in the dataset at 77.1%
3. *Embarked* column has very little missing values
4. Columns [PassengerId,Ticket,Fare] does'nt contribute the data analysis as they are highly categorical objects
5. Pclass is **miscategorised** as *numerical* but it should be *categorical*

---

- Removing column '**Cabin**' as the missing values in this column can disorient the intension of the model analysis
- Removig columns [PassengerId,Ticket,Fare] as they are highly categorical objects and does'nt contribute to target feature modelling

In [26]:
df=df.drop(columns=['Cabin','Ticket','PassengerId'])

In [27]:
df['Pclass']=df['Pclass'].astype('category')

In [28]:
num_cols=df.select_dtypes(include=[int,float]).columns
cat_cols=df.select_dtypes(include=[object,'category','string']).columns
print('Numerical Columns:',num_cols)
print('Categorical Columns:',cat_cols)

Numerical Columns: Index(['Survived', 'Age', 'SibSp', 'Parch', 'Fare'], dtype='str')
Categorical Columns: Index(['Pclass', 'Name', 'Sex', 'Embarked'], dtype='str')


## Handling Missing Values in Numerical Data

missing values in num_cols are handled using mean, median, mode inplace of missing values to assume the overall distribution of the data

In [29]:
df['Age']=df['Age'].fillna(df['Age'].median())


## Handling Missing Values in Categorical Data

In [30]:
df['Embarked']=df['Embarked'].fillna(df['Embarked'].mode()[0])
df['Sex']=df['Sex'].map({'male': 0, 'female': 1})

In [31]:
df.isnull().sum()

Survived    0
Pclass      0
Name        0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64

In [32]:
df.duplicated().sum()

np.int64(0)

In [33]:
df.head(10)

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,"Braund, Mr. Owen Harris",0,22.0,1,0,7.2500,S
1,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,38.0,1,0,71.2833,C
2,1,3,"Heikkinen, Miss. Laina",1,26.0,0,0,7.9250,S
3,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,35.0,1,0,53.1000,S
4,0,3,"Allen, Mr. William Henry",0,35.0,0,0,8.0500,S
5,0,3,"Moran, Mr. James",0,28.0,0,0,8.4583,Q
6,0,1,"McCarthy, Mr. Timothy J",0,54.0,0,0,51.8625,S
7,0,3,"Palsson, Master. Gosta Leonard",0,2.0,3,1,21.0750,S
8,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",1,27.0,0,2,11.1333,S
9,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",1,14.0,1,0,30.0708,C


In [34]:
df = pd.get_dummies(df, columns=['Embarked'], drop_first=True)

In [35]:
df.head(10)

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Fare,Embarked_Q,Embarked_S
0,0,3,"Braund, Mr. Owen Harris",0,22.0,1,0,7.2500,False,True
1,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,38.0,1,0,71.2833,False,False
2,1,3,"Heikkinen, Miss. Laina",1,26.0,0,0,7.9250,False,True
3,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,35.0,1,0,53.1000,False,True
4,0,3,"Allen, Mr. William Henry",0,35.0,0,0,8.0500,False,True
5,0,3,"Moran, Mr. James",0,28.0,0,0,8.4583,True,False
6,0,1,"McCarthy, Mr. Timothy J",0,54.0,0,0,51.8625,False,True
7,0,3,"Palsson, Master. Gosta Leonard",0,2.0,3,1,21.0750,False,True
8,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",1,27.0,0,2,11.1333,False,True
9,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",1,14.0,1,0,30.0708,False,False


In [36]:
df.to_csv('Titanic-Survival-Dataset/Data/processed/cleaned_data.csv', index=False)

# **Summary of Data Cleaning**
1. Irrelevant columns are removed
2. Correct DataTypes for each and every column
3. Handled Numerical missing values
4. Handled Categorical missing values
5. Encoded Categorical data
6. duplicate values are removed